# Smart Trip Budget Planner | Single Agent (ReAct)

In [1]:
import ast
import operator
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
# Web search tool (free, no API key; requires `duckduckgo-search` package)
search_tool = DuckDuckGoSearchResults(name="web_search")

In [4]:
# Safe arithmetic evaluator (no eval() -- prevents code injection)
SAFE_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.Pow: operator.pow, ast.USub: operator.neg,
}

In [5]:
def safe_eval(expr: str):
    """Safely evaluate arithmetic expressions only."""
    tree = ast.parse(expr, mode="eval")
    def _eval(node):
        if isinstance(node, ast.Expression):
            return _eval(node.body)
        elif isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        elif isinstance(node, ast.BinOp) and type(node.op) in SAFE_OPS:
            return SAFE_OPS[type(node.op)](_eval(node.left), _eval(node.right))
        elif isinstance(node, ast.UnaryOp) and type(node.op) in SAFE_OPS:
            return SAFE_OPS[type(node.op)](_eval(node.operand))
        else:
            raise ValueError(f"Unsupported expression: {ast.dump(node)}")
    return _eval(tree)

In [6]:
# Currency converter with static demo exchange rates
EXCHANGE_RATES = {
    ("USD", "EUR"): 0.92, ("EUR", "USD"): 1.09,
    ("USD", "GBP"): 0.79, ("GBP", "USD"): 1.27,
    ("USD", "JPY"): 149.50, ("JPY", "USD"): 0.0067,
    ("EUR", "GBP"): 0.86, ("GBP", "EUR"): 1.16,
    ("EUR", "JPY"): 162.70, ("JPY", "EUR"): 0.0061,
    ("GBP", "JPY"): 189.40, ("JPY", "GBP"): 0.0053,
}

In [7]:
# Tools

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. Example: '150 * 5 + 80 * 3'"""
    try:
        result = safe_eval(expression)
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error evaluating '{expression}': {e}"

@tool
def currency_converter(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount between currencies. Supported: USD, EUR, GBP, JPY."""
    from_c = from_currency.upper()
    to_c = to_currency.upper()
    if from_c == to_c:
        return f"{amount:.2f} {from_c} = {amount:.2f} {to_c}"
    rate = EXCHANGE_RATES.get((from_c, to_c))
    if not rate:
        return f"Conversion from {from_c} to {to_c} is not supported."
    converted = amount * rate
    return f"{amount:.2f} {from_c} = {converted:.2f} {to_c} (rate: {rate})"

In [8]:
# Create the agent
model = ChatOpenAI(model="gpt-4o")
agent = create_agent(
    model=model,
    tools=[search_tool, calculator, currency_converter],
    system_prompt=(
        "You are a smart trip budget planner. Help users estimate travel costs. "
        "Search the web for prices (flights, hotels, food), use the calculator for totals, "
        "and convert currencies as needed. Always provide a breakdown and final total."
    )
)

In [ ]:
# Plot the agent
plot_mermaid(agent)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> model;
	model -.-> __end__;
	model -.-> tools;
	tools -.-> model;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [10]:
# Run the agent
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Plan a 5-day trip budget from New York to Tokyo. Include flights, hotel, food, and transport."
    }]
})

print(result["messages"][-1].content)

Here's a breakdown of the estimated costs for a 5-day trip from New York to Tokyo:

1. **Flights**: Round-trip from New York to Tokyo costs approximately **$1300**.

2. **Accommodation**: Average hotel price is around $444 per night. For 5 nights, it totals to **$2500**.

3. **Food**: Average daily food cost in Tokyo is about $75 per day. For 5 days, this amounts to **$375**.

4. **Transportation**: Average daily cost for transport in Tokyo is estimated at $21 per day. For 5 days, this totals to **$105**.

**Total Estimated Cost**: **$4280**

This is the estimated cost for your trip, which covers flights, accommodation, food, and transport. Prices may vary, and it's always good to plan for some additional expenses.


In [11]:
# Streaming

stream_invoke(
    agent, {
        "messages": [{
            "role": "user",
            "content": "Plan a 5-day trip budget from New York to Tokyo. Include flights, hotel, food, and transport."
        }]
    })


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

┌─ HUMAN
│ Plan a 5-day trip budget from New York to Tokyo. Include flights, hotel, food, and transport.
└────────────────────────────────────────

┌─ AI
│ → tool: web_search({'query': 'round trip flight cost from New York to Tokyo'})
│ → tool: web_search({'query': 'average hotel price per night in Tokyo'})
│ → tool: web_search({'query': 'average daily food cost in Tokyo'})
│ → tool: web_search({'query': 'average daily transport cost in Tokyo'})
└────────────────────────────────────────

┌─ TOOL (web_search)
│ snippet: Averageround-tripprice for 1-stopflightsfromNewYorkJohn F Kennedy AirporttoTokyoNarita Airport clicked by KAYAK users in the last 12 months (only includestripswith the same layover airport on both legs)., title: $289 CHEAPFLIGHTSfromNewYorkJFKtoTokyoNarita... | KAYAK, link: https://www.ka

{'messages': [HumanMessage(content='Plan a 5-day trip budget from New York to Tokyo. Include flights, hotel, food, and transport.', additional_kwargs={}, response_metadata={}, id='12cce518-0cb8-4345-a42e-0f9ae812ad4a'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 214, 'total_tokens': 310, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_d13f8160b5', 'id': 'chatcmpl-DIpau8yYBHmTxf5UEuZAxYZPvTe8O', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ce5b8-8db3-7c91-a801-35a03a7cf96c-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'round trip flight cost from New York to Tokyo'}, 'id': 'call_r6keozVAaySzDLX